This notebook implements Andrej Karpathy's Autograd engine and an MLP in javascript, following his tutorial here: https://www.youtube.com/watch?v=VMj-3S1tku0

In [186]:
class Value {
    data: number
    _backward: () => void
    children: Set<Value>

    constructor(data: number, children: Value[] = [], op: String = "") {
        this.data = data
        this.children = new Set(children)
        this.op = op
        this.grad = 0
        this._backward = () => {}
    }

    add(other: Value) {
        const out = new Value(this.data + other.data, [this, other])

        out._backward = () => {
            this.grad += 1 * out.grad
            other.grad += 1 * out.grad
        }

        return out
    }

    mul(other: Value) {
        const out = new Value(this.data * other.data, [this, other])

        out._backward = () => {
            this.grad += other.data * out.grad
            other.grad += this.data * out.grad
        }

        return out
    }

    tanh() {
        // let x = this.data
        // let t = (Math.exp(2*x) - 1) / (Math.exp(2*x) + 1)

        // const out = new Value(
        //     t, [this]
        // )

        // out._backward = () => {
        //     this.grad += (1-t**2) * out.grad
        // }

        // return out

        let e = this.mul(new Value(2)).exp()

        return e.sub(new Value(1)).div(e.add(new Value(1)))
    }

    exp() {
        let x = this.data

        const out = new Value(
            Math.exp(x), [this]
        )

        out._backward = () => {
            this.grad += out.data * out.grad
        }

        return out
    }

    div(other: Value) {
        return this.mul(other.pow(-1))
    }

    pow(other: number) {
        if (typeof other !== 'number') throw new Exception("Value#pow takes a number.")
        
        const out = new Value(
            this.data ** other,
            [this]
        )

        out._backward = () => {
            this.grad += other * this.data ** (other-1) * out.grad
        }

        return out
    }

    backward() {
        const topo = []
        const visited = new Set()

        // DFS iteration
        function buildTopo(v: Value) {
            if (!visited.has(v)) {
                visited.add(v)
                
                for (const child of v.children) {
                    buildTopo(child)
                }
                topo.push(v)
            }
        }

        this.grad = 1
        buildTopo(this)
        topo.reverse()

        for (const value of topo) {
            value._backward()
        }
    }

    negate() {
        return this.mul(new Value(-1))
    }

    sub(other: Value) {
        return this.add(other.negate())
    }
    
    [Symbol.for("Deno.customInspect")]() {
        return `Value(data=${this.data.toFixed(2)}, grad=${this.grad.toFixed(2)})`
    }
}

function v(valueInitializer) { return new Value(valueInitializer) }

Symbol(Deno.customInspect)

In [121]:
let x1 = v(2)
let x2 = v(0)

let w1 = v(-3)
let w2 = v(1)

let b = v(6.8813735870195432)

let x1w1 = x1.mul(w1)
let x2w2 = x2.mul(w2)

let x1w1x2w2 = x1w1.add(x2w2)

let n = x1w1x2w2.add(b)

let o = n.tanh()

o.backward()

console.log(JSON.stringify({x1}, null, 2))

{
  "x1": {
    "data": 2,
    "children": {},
    "op": "",
    "grad": -1.5
  }
}


In [ ]:
v(2).div(v(4))

Value(data=-2.00, grad=0.00)

In [245]:
function random(min: number, max: number) {
    return Math.random() * (max - min) + min;
}

class Neuron {
    w: Value[]
    b: Value

    constructor(nin: number) {
        this.w = Array.from({ length: nin }).map(() => new Value(random(-1, 1)))
        this.b = new Value(random(-1, 1))
    }

    invoke(x) {
        // w * x + b
        let sum = this.b
        for (let i = 0; i < this.w.length; i++) {
            sum = sum.add(this.w[i].mul(x[i]))
        }
        return sum.tanh()
    }

    parameters(): Value[] {
        return [...this.w, this.b]
    }
}

class Layer {
    neurons: Neuron[]
    
    constructor(nin: number, nout: number) {
        this.neurons = Array.from({ length: nout }).map(() => new Neuron(nin))
    }

    invoke(x) {
        const outs = this.neurons.map(neuron => neuron.invoke(x))

        return outs.length === 1 ? outs[0] : outs
    }

    parameters() {
        return this.neurons.flatMap(x => x.parameters())
    }
}

class MLP {
    layerSizes: number[]
    layers: Layer[]
    
    constructor(nin: number, nouts: number[]) {
        this.layerSizes = [nin, ...nouts]
        this.layers = Array.from({length: nouts.length}).map((_x, i) => new Layer(this.layerSizes[i], this.layerSizes[i + 1]))
    }

    invoke(x: Value[]) {
        for (const layer of this.layers) {
            x = layer.invoke(x)
        }
        return x
    }

    parameters() {
        return this.layers.flatMap(x => x.parameters())
    }
}

In [238]:
const xs = [
    [2, 3, -1],
    [3, -1, 0.5],
    [.5, 1, 1],
    [1, 1, -1]
].map(x => x.map(y => new Value(y)))

const ys = [1, -1, -1, 1].map(x => new Value(x))

In [262]:
const mlp = new MLP(3, [4, 4, 1])

In [263]:
mlp.layers[0].neurons[0].w[0]

Value(data=1.00, grad=0.00)

In [264]:
for (let step = 0; step < 20; step++) {
    // forward pass
    const ypred = xs.map(x => mlp.invoke(x))

    // mean squared error loss
    let loss = v(0)

    for (let i = 0; i < ys.length; i++) {
        loss = loss.add(ypred[i].sub(ys[i]).pow(2))
    }

    console.log({ step, loss })
    
    // backward pass
    for (const p of mlp.parameters()) {
        p.grad = 0
    }
    loss.backward()

    // update
    for (const p of mlp.parameters()) {
        p.data += -0.05 * p.grad
    }
}

{ step: 0, loss: Value(data=3.15, grad=0.00) }
{ step: 1, loss: Value(data=2.35, grad=0.00) }
{ step: 2, loss: Value(data=1.70, grad=0.00) }
{ step: 3, loss: Value(data=1.15, grad=0.00) }
{ step: 4, loss: Value(data=0.76, grad=0.00) }
{ step: 5, loss: Value(data=0.51, grad=0.00) }
{ step: 6, loss: Value(data=0.36, grad=0.00) }
{ step: 7, loss: Value(data=0.27, grad=0.00) }
{ step: 8, loss: Value(data=0.21, grad=0.00) }
{ step: 9, loss: Value(data=0.17, grad=0.00) }
{ step: 10, loss: Value(data=0.15, grad=0.00) }
{ step: 11, loss: Value(data=0.12, grad=0.00) }
{ step: 12, loss: Value(data=0.11, grad=0.00) }
{ step: 13, loss: Value(data=0.09, grad=0.00) }
{ step: 14, loss: Value(data=0.08, grad=0.00) }
{ step: 15, loss: Value(data=0.08, grad=0.00) }
{ step: 16, loss: Value(data=0.07, grad=0.00) }
{ step: 17, loss: Value(data=0.06, grad=0.00) }
{ step: 18, loss: Value(data=0.06, grad=0.00) }
{ step: 19, loss: Value(data=0.05, grad=0.00) }


0.2809689553710428